In [1]:
# Activar autoreload para la recarga automática de módulos
%load_ext autoreload
%autoreload 2

Failed to read module file 'C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\shlex.py' for module 'shlex': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Users\arodriguez\Projects\meridian-mio\.venv_meridian\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\arodriguez\Projects\meridian-mio\.venv_meridian\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\importlib\__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_im

In [7]:
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "credentials.json"

In [2]:
import joblib
from meridian.analysis import optimizer
from meridian.analysis import summarizer

# Descargar el modelo y guardarlo en la carpeta 'comparison_metrics/' como 'model.pkl'
model = "model.pkl"
mmm = joblib.load(model)

In [3]:
# ========================
# IMPORTACIÓN DE PAQUETES
# ========================

# Numpy y Pandas: manipulación de datos numéricos y estructuras tabulares.
import numpy as np
np.random.seed(42)
import pandas as pd

# TensorFlow y TensorFlow Probability: base del modelo bayesiano en Meridian.
import tensorflow as tf
tf.random.set_seed(42)
import tensorflow_probability as tfp
tf.keras.backend.set_floatx('float32')

# ArviZ: análisis bayesiano y visualización de distribuciones a posteriori.
import arviz as az

# xarray: estructuras tipo Dataset multidimensional (usado en resultados de inferencia).
import xarray as xr

# IPython: utilidades para entorno interactivo (no se usa aún, pero útil para display).
import IPython

# ===============================
# MÓDULOS DE MERIDIAN (OFICIALES)
# ===============================

# Constantes internas de Meridian.
from meridian import constants

# Módulos de carga y preparación de datos.
from meridian.data import load
from meridian.data import test_utils
from meridian.data.input_data import InputData
from meridian.data import input_data  # útil para funciones auxiliares

# Módulos de modelado y especificación del modelo.
from meridian.model import model
from meridian.model import spec
from meridian.model import prior_distribution
from meridian.model.model import Meridian  # clase principal del modelo

# Módulos de análisis del modelo entrenado.
from meridian.analysis import optimizer     # optimización de presupuesto
from meridian.analysis import analyzer      # cálculo de métricas y efectos
from meridian.analysis import visualizer    # gráficos y visualizaciones
from meridian.analysis import summarizer    # resumen HTML del modelo
from meridian.analysis import formatter     # formato de métricas para reporting

# =============================
# UTILIDADES ADICIONALES
# =============================

# tqdm: barra de progreso útil en loops largos o procesos MCMC.
from tqdm import tqdm
import time

# matplotlib: visualización básica de datos.
import matplotlib.pyplot as plt

# =============================
# OCULTAR WARNINGS
# =============================

import warnings

# 🚫 Ignorar todos los warnings (general)
warnings.filterwarnings('ignore')


# ============================
# VERIFICACIÓN DEL ENTORNO
# ============================

# Verificar si hay GPU disponible y cuánta RAM tiene el entorno.
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9

print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))
print("Num CPUs Available: ", len(tf.config.experimental.list_physical_devices('CPU')))

from google.cloud import storage

Your runtime has 17.0 gigabytes of available RAM

Num GPUs Available:  0
Num CPUs Available:  1


# OPTIMIZACIÓN PRESUPUESTO

In [4]:
mmm.input_data.get_all_paid_channels()

array(['medios-digital-performance', 'medios-digital-branding',
       'medios-off', 'medios-patrocinio', 'medios-news-radio'],
      dtype=object)

In [5]:
num_paid_channels = mmm.input_data.get_all_paid_channels()

### Optimización liberando fronteras de pct relativo a la inversión histórica por canal

In [8]:
start_date_ = "2025-01-01"
end_date_ = "2025-09-01"     # cambia por la fecha real de fin
# ========================================
# OPTIMIZACIÓN PRESUPUESTARIA
# ========================================

# Crear el optimizador desde el modelo entrenado
budget_optimizer = optimizer.BudgetOptimizer(mmm)

#Reestricciones
num_canales = len(num_paid_channels)

# 1. Definir las restricciones inferiores
# Mover la inversión sin reestricciones salvo para envíos
lower_constraints = [1] * num_canales
upper_constraints = [1] * num_canales

# FIJAR EL GASTO DE ENVÍOS
lower_constraints[4] = 0
upper_constraints[4] = 0

# Ejecutar la optimización con todos los parámetros configurables
optimization_results = budget_optimizer.optimize(
    use_posterior=True,
    selected_times=None,
    fixed_budget=True, # Mantenemos el presupuesto total fijo
    budget=None, # None = usa el gasto histórico total
    start_date = start_date_,
    end_date = end_date_,
    # Restricciones Ajustadas
    spend_constraint_lower=lower_constraints,
    spend_constraint_upper=upper_constraints,
    target_roi=None,
    target_mroi=None,
    gtol=0.0001,
    use_optimal_frequency=True,
    use_kpi=True,
    confidence_level=0.9,
    batch_size=10000,
    use_pct_total_abs=True,
    pct_total_min=[0.1, 0.0, 0.1, 0.1, 0.0],
    pct_total_max=[0.7, 0.5, 0.6, 0.4, 0.5]
)

# ========================================
# EXPORTAR Y SUBIR RESULTADOS
# ========================================

import os

# 1️⃣ Crear el directorio temporal local
local_dir = "/tmp/optimization_report_meridian"
os.makedirs(local_dir, exist_ok=True)

output_filename = "optimization_output_free_boundaries.html"
local_path = os.path.join(local_dir, output_filename)

# 2️⃣ Generar el resumen de optimización en local
optimization_results.output_optimization_summary(
    filename=output_filename,
    filepath=local_dir
)

# Abrir en Colab
IPython.display.HTML(filename=f"{local_dir}/{output_filename}")


✅ Report saved to /tmp/optimization_report_meridian\optimization_output_free_boundaries.html


Channel,Non-optimized spend,Optimized spend
medios-patrocinio,33%,29%
medios-digital-branding,20%,21%
medios-digital-performance,9%,20%
medios-off,30%,18%
medios-news-radio,8%,12%
